# Phase 9a — ds004789 Data Loader (Bipolar Channel Mapping + Word Epochs)

Extracts UPenn RAM (`ds004789`) free recall data directly from OpenNeuro/S3:
- Resolves bipolar channel pairs (`c1-c2`) against electrode contact anatomy.
- Epochs `trial_type == 'WORD'` encoding trials and determines recall labels.
- Extracts 5-band log power (`theta`, `alpha`, `beta`, `lgamma`, `hgamma`).
- Generates `X_fr1.npy` and `M_fr1.csv`.


In [1]:
import subprocess, sys, os, time, json, re, warnings, pathlib
import urllib.request, io
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'mne'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
import mne
mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')

WORK    = pathlib.Path('/kaggle/working')
RAW     = WORK / 'ds004789'
CACHE   = WORK / 'cache_fr1'
OUT_DIR = WORK / 'phase9a_out'

for d in [RAW, CACHE, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('mne version :', mne.__version__)
print('RAW dir     :', RAW)
print('CACHE dir   :', CACHE)
print('OUT dir     :', OUT_DIR)


mne version : 1.13.2
RAW dir     : /kaggle/working/ds004789
CACHE dir   : /kaggle/working/cache_fr1
OUT dir     : /kaggle/working/phase9a_out


In [2]:
print('Fetching dataset manifest from OpenNeuro GraphQL API...')
t0 = time.time()

manifest_cache = CACHE / 'dataset_manifest.json'
if manifest_cache.exists():
    with open(manifest_cache, 'r', encoding='utf-8') as f:
        files = json.load(f)
    print(f'Loaded cached manifest: {len(files)} files ({time.time()-t0:.1f}s)')
else:
    query = 'query { dataset(id: "ds004789") { latestSnapshot { files(recursive: true) { filename urls size } } } }'
    req = urllib.request.Request(
        'https://openneuro.org/crn/graphql',
        data=json.dumps({'query': query}).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'User-Agent': 'Mozilla/5.0'}
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read())
    files = data['data']['dataset']['latestSnapshot']['files']
    with open(manifest_cache, 'w', encoding='utf-8') as f:
        json.dump(files, f)
    print(f'Fetched {len(files)} files in {time.time()-t0:.1f}s and cached to disk.')

file_map = {f['filename']: f for f in files}
print('Unique files in manifest:', len(file_map))


Fetching dataset manifest from OpenNeuro GraphQL API...
Fetched 7427 files in 2.1s and cached to disk.
Unique files in manifest: 7427


In [3]:
def download_file_direct(rel_path, target_path):
    if target_path.exists() and target_path.stat().st_size > 0:
        return True
    info = file_map.get(rel_path)
    if not info or not info.get('urls'):
        return False
    url = info['urls'][0]
    target_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=30) as r, open(target_path, 'wb') as f:
        f.write(r.read())
    return True

# Download participants.tsv & description
for rf in ['participants.tsv', 'participants.json', 'dataset_description.json']:
    if rf in file_map:
        download_file_direct(rf, RAW / rf)

# Identify and download all electrodes and events .tsv files
meta_files = [
    f['filename'] for f in files
    if f['filename'].endswith('_electrodes.tsv') or f['filename'].endswith('_events.tsv')
]
print(f'Downloading {len(meta_files)} metadata files in parallel...')
t0 = time.time()
def fetch_worker(rel_p):
    try:
        return download_file_direct(rel_p, RAW / rel_p)
    except Exception:
        return False

with ThreadPoolExecutor(max_workers=16) as pool:
    results = list(pool.map(fetch_worker, meta_files))

print(f'Downloaded {sum(1 for r in results if r)} files in {time.time()-t0:.1f}s')


Downloaded 1106 files in 27.6s


In [4]:
# CORE regions regex mapping
RX = {
    'hippocampus'    : re.compile(r'hipp|CA[123]|subicul|dentate', re.I),
    'amygdala'       : re.compile(r'amyg', re.I),
    'parahippocampal': re.compile(r'parahipp|perirhinal|entorhinal|PHG', re.I),
    'lateral_temporal': re.compile(r'(middle|superior|inferior)\s*temporal|MTG|STG|ITG|lateral.*temp', re.I),
}

def label_to_region(label):
    if not isinstance(label, str):
        return None
    for reg, rx in RX.items():
        if rx.search(label):
            return reg
    return None

def contact_region(row, cols=('ind.region','region','Loc_label','label','name')):
    for c in cols:
        if c in row.index and pd.notna(row[c]):
            r = label_to_region(str(row[c]))
            if r:
                return r
    return None

def resolve_channel_region(ch_name, contact_map):
    """
    Maps a channel name like '1Ld1-1Ld2' to a CORE region.
    """
    ch_clean = str(ch_name).strip()
    if '-' in ch_clean:
        c1, c2 = ch_clean.split('-', 1)
        c1, c2 = c1.strip(), c2.strip()
        r1 = contact_map.get(c1)
        r2 = contact_map.get(c2)
        if r1 and r2 and r1 == r2:
            return r1
        return r1 or r2
    return contact_map.get(ch_clean)

print('Region mapping logic verified.')


Region mapping logic verified.


In [5]:
CORE = ['hippocampus', 'amygdala', 'parahippocampal', 'lateral_temporal']
elec_all = list(RAW.rglob('*_electrodes.tsv'))
rows = []

for ef in elec_all:
    parts = ef.parts
    sub = next((p for p in parts if p.startswith('sub-')), None)
    ses = next((p for p in parts if p.startswith('ses-')), 'ses-0')
    if not sub:
        continue
    try:
        df = pd.read_csv(ef, sep='\t', low_memory=False)
    except Exception:
        continue
    df['_reg'] = df.apply(contact_region, axis=1)
    matched = df[df['_reg'].notna()]
    counts = matched['_reg'].value_counts().to_dict()
    row = {'sub': sub, 'ses': ses}
    for c in CORE:
        row[c] = counts.get(c, 0)
    rows.append(row)

plan = pd.DataFrame(rows).fillna(0)
if not plan.empty:
    plan['n_core'] = plan[CORE].sum(axis=1).astype(int)
    plan = plan.sort_values('n_core', ascending=False).drop_duplicates(subset=['sub', 'ses'])
    plan_ok = plan[plan['n_core'] > 0].copy()
else:
    plan_ok = pd.DataFrame(columns=['sub', 'ses', 'n_core'] + CORE)

print(f'Total sessions planned with CORE contacts: {len(plan_ok)}')
plan_ok.to_csv(CACHE / 'plan_sessions.csv', index=False)
print(plan_ok.head(5).to_string(index=False))


Total sessions planned with CORE contacts: 517
       sub   ses  hippocampus  amygdala  parahippocampal  lateral_temporal  n_core
sub-R1405E ses-1            6         0                6                89     101
sub-R1334T ses-1            2         0                6                80      88
sub-R1334T ses-0            2         0                6                80      88
sub-R1334T ses-2            2         0                6                80      88
sub-R1385E ses-1            0         0                4                83      87


In [6]:
from scipy.signal import butter, sosfiltfilt

SFREQ_TARGET = 200
EPOCH_START  = -0.5
EPOCH_END    = 1.5
BANDS = {
    'theta' : (4,  8),
    'alpha' : (8, 13),
    'beta'  : (13, 30),
    'lgamma': (30, 70),
    'hgamma': (70, 120),
}

def butter_bandpass(lo, hi, fs, order=4):
    nyq = fs / 2.0
    sos = butter(order, [lo/nyq, min(hi/nyq, 0.99)], btype='band', output='sos')
    return sos

def band_power(sig, fs, lo, hi):
    sos = butter_bandpass(lo, hi, fs)
    filt = sosfiltfilt(sos, sig, axis=-1)
    return np.log(np.mean(filt**2, axis=-1) + 1e-12)

def epoch_logpower(raw_arr, fs, onsets_s, bands=BANDS, t_start=EPOCH_START, t_end=EPOCH_END):
    n_ch, n_t = raw_arr.shape
    n_pre  = int(-t_start * fs)
    n_post = int(t_end * fs)
    n_ep   = len(onsets_s)
    n_bands = len(bands)
    out = np.full((n_ep, n_ch, n_bands), np.nan)
    band_items = list(bands.items())
    for ei, onset in enumerate(onsets_s):
        s0 = int(onset * fs) - n_pre
        s1 = int(onset * fs) + n_post
        if s0 < 0 or s1 > n_t:
            continue
        seg = raw_arr[:, s0:s1]
        for bi, (bname, (lo, hi)) in enumerate(band_items):
            out[ei, :, bi] = band_power(seg, fs, lo, hi)
    return out

def zscore_pairs(X):
    mu = np.nanmean(X, axis=0, keepdims=True)
    sd = np.nanstd(X, axis=0, keepdims=True) + 1e-8
    return (X - mu) / sd

print('Signal processing functions initialized.')


Signal processing functions initialized.


In [7]:
def fetch_edf_direct(sub, ses):
    prefix = f'{sub}/{ses}/ieeg/'
    candidates = [
        f for f in files
        if f['filename'].startswith(prefix) and f['filename'].endswith('.edf')
    ]
    if not candidates:
        return None
    bipolar = [c for c in candidates if 'bipolar' in c['filename']]
    selected = bipolar[0] if bipolar else candidates[0]

    rel_p = selected['filename']
    target = RAW / rel_p
    if target.exists() and target.stat().st_size > 0:
        return target

    print(f'    Downloading {rel_p} ({selected["size"]/1e6:.1f} MB)...', end=' ', flush=True)
    t_dl = time.time()
    target.parent.mkdir(parents=True, exist_ok=True)
    url = selected['urls'][0]
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r, open(target, 'wb') as f:
        while True:
            chunk = r.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
    print(f'Done ({time.time()-t_dl:.1f}s)')
    return target

def drop_edf_direct(edf_path):
    try:
        if edf_path and edf_path.exists():
            edf_path.unlink()
    except Exception:
        pass

print('fetch_edf_direct and drop_edf_direct ready.')


fetch_edf_direct and drop_edf_direct ready.


In [8]:
def process_session(sub, ses):
    cache_x = CACHE / f'{sub}_{ses}_X.npy'
    cache_m = CACHE / f'{sub}_{ses}_M.csv'
    if cache_x.exists() and cache_m.exists():
        X = np.load(cache_x)
        return True, len(X)

    # Locate electrodes
    elec_matches = list((RAW / sub / ses).rglob('*_electrodes.tsv'))
    if not elec_matches:
        return False, 0
    elec_df = pd.read_csv(elec_matches[0], sep='\t', low_memory=False)

    # Locate events
    evt_matches = list((RAW / sub / ses).rglob('*_events.tsv'))
    if not evt_matches:
        return False, 0
    events_df = pd.read_csv(evt_matches[0], sep='\t', low_memory=False)

    # Filter to WORD encoding events only
    if 'trial_type' in events_df.columns:
        word_events = events_df[events_df['trial_type'] == 'WORD'].copy()
    else:
        word_events = events_df.copy()

    if word_events.empty or 'onset' not in word_events.columns:
        return False, 0

    # Compute recalled column if not present
    if 'recalled' not in word_events.columns:
        rec_words = set(
            events_df[events_df['trial_type'] == 'REC_WORD']['item_name']
            .dropna().str.upper().tolist()
        )
        word_events['recalled'] = (
            word_events['item_name'].astype(str).str.upper().isin(rec_words).astype(int)
        )

    # Build contact -> region map
    if 'name' not in elec_df.columns and 'channel' in elec_df.columns:
        elec_df = elec_df.rename(columns={'channel': 'name'})
    elec_df['_reg'] = elec_df.apply(contact_region, axis=1)
    contact_map = dict(zip(elec_df['name'].astype(str), elec_df['_reg']))

    # Download EDF
    edf_file = fetch_edf_direct(sub, ses)
    if not edf_file:
        return False, 0

    try:
        raw_mne = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
        if raw_mne.info['sfreq'] != SFREQ_TARGET:
            raw_mne.resample(SFREQ_TARGET, npad='auto')
        fs = raw_mne.info['sfreq']

        # Map channels (pairs or single contacts) to CORE regions
        ch_names = raw_mne.ch_names
        ch_map = {c: resolve_channel_region(c, contact_map) for c in ch_names}
        core_chs = [c for c in ch_names if ch_map.get(c) in CORE]

        if not core_chs:
            print(f'    [Debug] No channel matches CORE in {sub}/{ses}')
            drop_edf_direct(edf_file)
            return False, 0

        raw_pick = raw_mne.copy().pick(core_chs)
        data = raw_pick.get_data()
        onsets = word_events['onset'].dropna().values.astype(float)

        X_ep = epoch_logpower(data, fs, onsets)
        valid = ~np.isnan(X_ep).all(axis=(1, 2))
        X_ep = X_ep[valid]
        ev_valid = word_events.iloc[np.where(valid)[0]].reset_index(drop=True)

        if len(X_ep) == 0:
            drop_edf_direct(edf_file)
            return False, 0

        # Aggregate channels per CORE region
        ch_regs = [ch_map.get(c) for c in core_chs]
        region_X = []
        for reg in CORE:
            idx = [i for i, r in enumerate(ch_regs) if r == reg]
            if idx:
                region_X.append(X_ep[:, idx, :].mean(axis=1))
            else:
                region_X.append(np.zeros((len(X_ep), len(BANDS))))
        X_reg = np.stack(region_X, axis=1)  # (N_trials, 4, 5)

        meta = ev_valid[['onset']].copy()
        for col in ['recalled', 'serialpos', 'item_name', 'list', 'subject']:
            if col in ev_valid.columns:
                meta[col] = ev_valid[col].values
            else:
                meta[col] = np.nan

        X_reg = zscore_pairs(X_reg)
        np.save(cache_x, X_reg)
        meta.to_csv(cache_m, index=False)
        drop_edf_direct(edf_file)
        return True, len(X_reg)
    except Exception as e:
        print(f'    Processing error: {e}')
        drop_edf_direct(edf_file)
        return False, 0

print('process_session logic compiled successfully.')


process_session logic compiled successfully.


In [9]:
MAX_SESSIONS = 30
TIME_BUDGET  = 5400  # 90 minutes

plan_path = CACHE / 'plan_sessions.csv'
plan_df = pd.read_csv(plan_path)
plan_df = plan_df.sort_values('n_core', ascending=False).head(MAX_SESSIONS)
print(f'Starting extraction for up to {len(plan_df)} sessions...')

results = []
t_loop = time.time()

for i, row in plan_df.iterrows():
    if time.time() - t_loop > TIME_BUDGET:
        print('Time budget reached, finalizing cached sessions...')
        break
    sub = str(row['sub'])
    ses = str(row['ses'])

    cache_x = CACHE / f'{sub}_{ses}_X.npy'
    cache_m = CACHE / f'{sub}_{ses}_M.csv'
    if cache_x.exists() and cache_m.exists():
        X_tmp = np.load(cache_x)
        results.append({'sub': sub, 'ses': ses, 'n': len(X_tmp), 'status': 'cached'})
        print(f'[{len(results)}/{len(plan_df)}] {sub}/{ses} -> CACHED ({len(X_tmp)} trials)')
        continue

    print(f'[{len(results)+1}/{len(plan_df)}] {sub}/{ses} -> processing...')
    ok, n_trials = process_session(sub, ses)
    status = 'ok' if ok else 'failed'
    results.append({'sub': sub, 'ses': ses, 'n': n_trials, 'status': status})
    print(f'    Status: {status} ({n_trials} trials)')

res_df = pd.DataFrame(results)
if not res_df.empty and 'status' in res_df.columns:
    print('\n--- Session Summary ---')
    print(res_df['status'].value_counts().to_string())
    total_trials = res_df[res_df['status'].isin(['ok', 'cached'])]['n'].sum()
    print(f'Total trials extracted: {total_trials}')
else:
    print('No sessions processed.')


Starting extraction for up to 30 sessions...
[1/30] sub-R1405E/ses-1 -> processing...
    Status: ok (156 trials)
[2/30] sub-R1334T/ses-0 -> processing...
    Status: ok (312 trials)
[3/30] sub-R1334T/ses-1 -> processing...
    Status: ok (312 trials)
[4/30] sub-R1334T/ses-2 -> processing...
    Status: ok (180 trials)
[5/30] sub-R1385E/ses-1 -> processing...
    Status: ok (156 trials)
[6/30] sub-R1385E/ses-2 -> processing...
    Status: ok (156 trials)
[7/30] sub-R1385E/ses-3 -> processing...
    Status: ok (156 trials)
[8/30] sub-R1420T/ses-0 -> processing...
    Status: ok (156 trials)
[9/30] sub-R1420T/ses-1 -> processing...
    Status: ok (156 trials)
[10/30] sub-R1571T/ses-0 -> processing...
    Status: ok (300 trials)
[11/30] sub-R1329T/ses-1 -> processing...
    Status: ok (240 trials)
[12/30] sub-R1234D/ses-1 -> processing...
    Status: ok (300 trials)
[13/30] sub-R1234D/ses-0 -> processing...
    Status: ok (300 trials)
[14/30] sub-R1415T/ses-0 -> processing...
    Status: 

In [10]:
Xs, Ms = [], []
for xf in sorted(CACHE.glob('*_X.npy')):
    stem = xf.name[:-6]  # remove _X.npy
    mf = CACHE / f'{stem}_M.csv'
    if mf.exists():
        Xs.append(np.load(xf))
        Ms.append(pd.read_csv(mf))

if Xs:
    X_all = np.concatenate(Xs, axis=0)
    M_all = pd.concat(Ms, ignore_index=True)
    print(f'X_fr1 shape: {X_all.shape}  (trials, 4 regions, 5 bands)')
    print(f'M_fr1 rows : {len(M_all)}')
    np.save(OUT_DIR / 'X_fr1.npy', X_all)
    M_all.to_csv(OUT_DIR / 'M_fr1.csv', index=False)
    print('Saved tensors successfully to:', OUT_DIR)
else:
    print('WARNING: No sessions completed extraction yet.')


X_fr1 shape: (6122, 4, 5)  (trials, 4 regions, 5 bands)
M_fr1 rows : 6122
Saved tensors successfully to: /kaggle/working/phase9a_out


In [11]:
x_path = OUT_DIR / 'X_fr1.npy'
m_path = OUT_DIR / 'M_fr1.csv'

if x_path.exists() and m_path.exists():
    X_check = np.load(x_path)
    M_check = pd.read_csv(m_path)
    print('Tensor verification:')
    print('  X shape:', X_check.shape)
    print('  M shape:', M_check.shape)
    print('  NaN fraction:', round(float(np.isnan(X_check).mean()), 5))
    if 'recalled' in M_check.columns:
        print('  Mean recall rate:', round(float(M_check['recalled'].mean()), 3))
    print('PHASE 9a VERIFICATION SUCCESSFUL!')
else:
    print('Output tensors not present.')


Tensor verification:
  X shape: (6122, 4, 5)
  M shape: (6122, 6)
  NaN fraction: 0.0
  Mean recall rate: 0.244
PHASE 9a VERIFICATION SUCCESSFUL!


In [12]:
print('Phase 9a directory contents:')
for f in OUT_DIR.iterdir():
    print(f'  {f.name:20s} {f.stat().st_size/1e6:6.2f} MB')
print('\nPhase 9a extraction finished cleanly.')


Phase 9a directory contents:
  M_fr1.csv              0.17 MB
  X_fr1.npy              0.98 MB

Phase 9a extraction finished cleanly.
